# US-004 - EDA Base 2

Analise exploratoria da base operacional de compra. Como a Base 2 nao contem `perfil_latente`, a EDA junta `perfil_latente` e comportamento de manutencao da Base 1 por `id_cliente` apenas para analise exploratoria por grupo.

In [ ]:
import matplotlib
matplotlib.use('Agg')

import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

os.makedirs('reports', exist_ok=True)
sns.set_theme(style='whitegrid')

## Carregamento e join exploratorio

In [ ]:
base2 = pd.read_csv('data/raw/ford_clientes_operacional_compra.csv')
base1_cols = [
    'id_cliente',
    'perfil_latente',
    'fez_primeira_revisao_rede',
    'meses_ate_primeira_revisao',
    'perdeu_primeira_revisao',
    'voltou_tarde_revoltado',
    'trouxe_oleo_externo',
    'pede_desconto_revisao',
    'sensibilidade_desconto_pos',
    'qtde_revisoes_24m',
    'share_revisoes_rede_24m',
    'gasto_manutencao_rede_24m',
    'satisfacao_marca_24m',
    'churn_rede_24m',
]
base1_perfil = pd.read_csv('data/raw/ford_clientes_historico_completo.csv', usecols=base1_cols)
df = base2.merge(base1_perfil, on='id_cliente', how='inner')

print(f'Base 2 original: {base2.shape[0]:,} linhas, {base2.shape[1]:,} colunas')
print(f'Base 2 enriquecida para EDA: {df.shape[0]:,} linhas, {df.shape[1]:,} colunas')
df.head()

In [ ]:
missing_values = (
    base2.isna()
    .sum()
    .to_frame('missing_qtd')
    .assign(missing_pct=lambda x: (x['missing_qtd'] / len(base2) * 100).round(2))
    .sort_values(['missing_pct', 'missing_qtd'], ascending=False)
)

missing_values

## Plano de manutencao

`plano_manutencao` e derivado para esta EDA a partir do uso da rede Ford em 24 meses: clientes com alto share de revisoes na rede e pelo menos duas revisoes entram como `com_plano`; clientes com uso intermediario entram como `avulso_rede`; os demais como `fora_rede`.

In [ ]:
df['plano_manutencao'] = np.select(
    [
        df['share_revisoes_rede_24m'].ge(0.75) & df['qtde_revisoes_24m'].ge(2),
        df['share_revisoes_rede_24m'].ge(0.40),
    ],
    ['com_plano', 'avulso_rede'],
    default='fora_rede',
)

plano_por_perfil = pd.crosstab(
    df['perfil_latente'],
    df['plano_manutencao'],
    normalize='index',
).mul(100).round(2)

plano_por_perfil

In [ ]:
plano_plot = plano_por_perfil.reset_index().melt(
    id_vars='perfil_latente',
    var_name='plano_manutencao',
    value_name='percentual',
)

plt.figure(figsize=(9, 5))
ax = sns.barplot(
    data=plano_plot,
    x='perfil_latente',
    y='percentual',
    hue='plano_manutencao',
)
ax.set_title('Plano de manutencao por perfil latente')
ax.set_xlabel('Perfil latente')
ax.set_ylabel('Percentual de clientes')
plt.legend(title='Plano')
plt.tight_layout()
plt.savefig('reports/base2_plano_manutencao_por_perfil.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()

## Coeficiente de variacao entre grupos de perfil

O coeficiente de variacao abaixo mede quanto a media de cada feature varia entre os grupos de `perfil_latente`. Para variaveis categoricas de plano, a media representa a taxa de clientes em cada categoria.

In [ ]:
numeric_features = [
    'idade',
    'renda_mensal',
    'score_credito',
    'preco_veiculo',
    'valor_financiado',
    'entrada_pct',
    'km_estimado_ano',
    'tempo_habilitacao_anos',
    'distancia_concessionaria_km',
    'tempo_decisao_dias',
    'fez_primeira_revisao_rede',
    'perdeu_primeira_revisao',
    'voltou_tarde_revoltado',
    'trouxe_oleo_externo',
    'pede_desconto_revisao',
    'sensibilidade_desconto_pos',
    'qtde_revisoes_24m',
    'share_revisoes_rede_24m',
    'gasto_manutencao_rede_24m',
    'satisfacao_marca_24m',
    'churn_rede_24m',
]

media_por_perfil = df.groupby('perfil_latente')[numeric_features].mean(numeric_only=True)
cv_numeric = (media_por_perfil.std() / media_por_perfil.mean().replace(0, np.nan)).abs()

plano_dummies = pd.get_dummies(
    df[['perfil_latente', 'plano_manutencao']],
    columns=['plano_manutencao'],
    dtype=int,
)
taxa_plano_por_perfil = plano_dummies.groupby('perfil_latente').mean()
cv_plano = (taxa_plano_por_perfil.std() / taxa_plano_por_perfil.mean().replace(0, np.nan)).abs()

coeficiente_variacao = (
    pd.concat([cv_numeric, cv_plano])
    .sort_values(ascending=False)
    .rename('coeficiente_variacao')
    .reset_index()
    .rename(columns={'index': 'feature'})
)

coeficiente_variacao.head(15)

In [ ]:
top_discriminadores = coeficiente_variacao.head(10).copy()

plt.figure(figsize=(9, 5))
ax = sns.barplot(
    data=top_discriminadores,
    y='feature',
    x='coeficiente_variacao',
    color='#003478',
)
ax.set_title('Top discriminadores por coeficiente de variacao entre perfis')
ax.set_xlabel('Coeficiente de variacao')
ax.set_ylabel('Feature')
plt.tight_layout()
plt.savefig('reports/base2_top_discriminadores_cv.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()

maior_discriminador = coeficiente_variacao.iloc[0]
print(
    'Maior discriminador:',
    maior_discriminador['feature'],
    f"CV={maior_discriminador['coeficiente_variacao']:.3f}",
)
assert maior_discriminador['feature'].startswith('plano_manutencao')

top_discriminadores